# Function 7: Sometimes Lazy is Best
You are now optimising six hyper-parameters of a machine learning model. Note that it is a popular and frequently used model, so maybe you could search to see if anyone else has optisized it before?

In [1]:
import numpy as np
import pandas as pd

In [2]:
def load_inputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    # Make the file a proper list of lists
    content = "[" + content.replace("]\n[", "],[") + "]"

    # Safe eval with restricted globals
    return eval(content, {"array": np.array})


def load_outputs(file_path):
    with open(file_path, "r") as f:
        content = f.read()

    content = "[" + content.replace("]\n[", "],[") + "]"

    return eval(content, {"np": np})


def get_input_points(function_number, file_path="../inputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_inputs(file_path)
    index = function_number - 1

    inputs = [dataset[index] for dataset in data]
    return np.array(inputs)


def get_output_points(function_number, file_path="../outputs.txt"):
    if not 1 <= function_number <= 8:
        raise ValueError("Function number must be between 1 and 8")

    data = load_outputs(file_path)
    index = function_number - 1

    outputs = [row[index] for row in data]
    return np.array(outputs)

# Load inputs
X = np.load(r'initial_inputs.npy')
y = np.load(r'initial_outputs.npy')


# Get input and outputs from submissions
inputs_array = get_input_points(7)
outputs_array = get_output_points(7)


# Append inputs_f1_array to X
X = np.vstack((X, inputs_array))

# Append outputs_f1_array to Y
y = np.hstack((y, outputs_array))

print("New shape of X:", X.shape)
print("New shape of Y:", y.shape)

New shape of X: (37, 6)
New shape of Y: (37,)


In [3]:
def generate_nd_grid(max_points, dimensions):
    # define range for input
	r_min, r_max = 0, 1.0

	# generate a random sample from the domain (dimensions)
	nd_grid = r_min + np.random.rand(max_points, dimensions) * (r_max - r_min)

	return np.array(nd_grid)

In [4]:
max_points = 80000000
dimensions = 6  # Change this to the desired number of dimensions
X_grid = []
X_grid = generate_nd_grid(max_points, dimensions)
X_grid

array([[0.95976286, 0.9257647 , 0.53050616, 0.47690132, 0.5306655 ,
        0.41673173],
       [0.59447196, 0.1463849 , 0.37608024, 0.20130567, 0.83278835,
        0.99305883],
       [0.62728048, 0.91137047, 0.99582639, 0.83437686, 0.41695421,
        0.27627373],
       ...,
       [0.78688247, 0.52303223, 0.05582851, 0.0536819 , 0.34823268,
        0.73366963],
       [0.75772659, 0.45022058, 0.59630557, 0.30669121, 0.86905569,
        0.68743408],
       [0.03037212, 0.164384  , 0.86169527, 0.59530271, 0.15453135,
        0.76308239]])

In [5]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.optimize import fmin_l_bfgs_b

# Define a custom optimizer function
def custom_optimizer(obj_func, initial_theta, bounds):
    x, f, _ = fmin_l_bfgs_b(
        func=obj_func,
        x0=initial_theta,
        bounds=bounds,
        maxiter=1000,
        maxfun=1500
    )
    return x, f  # Only return what sklearn expects

# # 1. Define kernel and GP
# Set up kernel with wider bounds to avoid hitting upper limit
kernel = ConstantKernel(1.0, (1e-2, 1e5)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e3))

# Gaussian Process with custom optimizer
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,
    optimizer=custom_optimizer,
    normalize_y=True
)

# 2. Standardize data
scaler_X = StandardScaler()
scaler_Y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_Y.fit_transform(y.reshape(-1, 1)).ravel()

# 3. Fit GP model
gp.fit(X_scaled, y_scaled)

# 4. Define acquisition function (Expected Improvement)
def expected_improvement(x_unscaled, gp, xi=0.01):
    # Scale the input
    x = scaler_X.transform(x_unscaled.reshape(1, -1))
    mu, sigma = gp.predict(x, return_std=True)
    mu_sample_opt = np.max(gp.predict(gp.X_train_))

    with np.errstate(divide='warn'):
        imp = mu - mu_sample_opt - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei[0]  # return scalar

# 5. Find next point (optimize EI in unscaled space)
res = minimize(lambda x: -expected_improvement(x, gp),
               x0=np.random.uniform(0, 1, size=X.shape[1]),
               bounds=[(0, 1)] * X.shape[1],
               method='L-BFGS-B')

next_point = res.x

formatted_next_query = f"{next_point[0]:.6f}-{next_point[1]:.6f}-{next_point[2]:.6f}-{next_point[3]:.6f}-{next_point[4]:.6f}-{next_point[5]:.6f}"
print("Next point to evaluate (unscaled):", formatted_next_query)


Next point to evaluate (unscaled): 0.490217-0.231689-0.723120-0.960651-0.231874-0.803486
